In [0]:
archivos = dbutils.fs.ls("/Volumes/workspace/default/ventas_raw")

for archivo in archivos:
    print(archivo.name)

Clientes.csv
Metas.csv
Productos.csv
Sucursales.csv
Vendedores.csv
Ventas.csv


In [0]:
ruta_clientes = "/Volumes/workspace/default/ventas_raw/Clientes.csv"

df_clientes = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(ruta_clientes)
)

display(df_clientes)

ClienteCodigo,ClienteNombre,Rubro,Segmento,Comuna
CLI0001,Comercial Andina 001,Tecnología,Mediana Empresa,Quilicura
CLI0002,Comercial Andina 002,Tecnología,Mediana Empresa,Quilicura
CLI0003,Comercial Sur 003,Construcción,Mediana Empresa,Providencia
CLI0004,Distribuidora Central 004,Alimentos,PYME,Maipú
CLI0005,Soluciones Empresariales 005,Logística,Corporativo,Las Condes
CLI0006,Distribuidora Central 006,Logística,Corporativo,Puente Alto
CLI0007,Comercial Los Andes 007,Retail,Mediana Empresa,Santiago
CLI0008,Importadora Norte 008,Retail,Corporativo,Providencia
CLI0009,Comercial Los Andes 009,Manufactura,PYME,Puente Alto
CLI0010,Comercial Horizonte 010,Servicios,Mediana Empresa,Quilicura


In [0]:
df_clientes.printSchema()

root
 |-- ClienteCodigo: string (nullable = true)
 |-- ClienteNombre: string (nullable = true)
 |-- Rubro: string (nullable = true)
 |-- Segmento: string (nullable = true)
 |-- Comuna: string (nullable = true)



In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")

DataFrame[]

In [0]:
(
    df_clientes.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.bronze.clientes")
)

In [0]:
display(
    spark.sql("""
        SELECT *
        FROM workspace.bronze.clientes
        LIMIT 20
    """)
)

ClienteCodigo,ClienteNombre,Rubro,Segmento,Comuna
CLI0001,Comercial Andina 001,Tecnología,Mediana Empresa,Quilicura
CLI0002,Comercial Andina 002,Tecnología,Mediana Empresa,Quilicura
CLI0003,Comercial Sur 003,Construcción,Mediana Empresa,Providencia
CLI0004,Distribuidora Central 004,Alimentos,PYME,Maipú
CLI0005,Soluciones Empresariales 005,Logística,Corporativo,Las Condes
CLI0006,Distribuidora Central 006,Logística,Corporativo,Puente Alto
CLI0007,Comercial Los Andes 007,Retail,Mediana Empresa,Santiago
CLI0008,Importadora Norte 008,Retail,Corporativo,Providencia
CLI0009,Comercial Los Andes 009,Manufactura,PYME,Puente Alto
CLI0010,Comercial Horizonte 010,Servicios,Mediana Empresa,Quilicura


In [0]:
archivos_bronze = {
    "productos": "Productos.csv",
    "vendedores": "Vendedores.csv",
    "sucursales": "Sucursales.csv",
    "ventas": "Ventas.csv",
    "metas": "Metas.csv"
}

ruta_base = "/Volumes/workspace/default/ventas_raw"

for tabla, archivo in archivos_bronze.items():

    ruta_archivo = f"{ruta_base}/{archivo}"

    df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(ruta_archivo)
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"workspace.bronze.{tabla}")
    )

    print(f"Tabla creada: workspace.bronze.{tabla}")

Tabla creada: workspace.bronze.productos
Tabla creada: workspace.bronze.vendedores
Tabla creada: workspace.bronze.sucursales
Tabla creada: workspace.bronze.ventas
Tabla creada: workspace.bronze.metas


In [0]:
display(
    spark.sql("""
        SHOW TABLES IN workspace.bronze
    """)
)

database,tableName,isTemporary
bronze,clientes,false
bronze,metas,false
bronze,productos,false
bronze,sucursales,false
bronze,vendedores,false
bronze,ventas,false
